In [1]:
import os
import pandas as pd
import pyarrow.parquet as pq

In [ ]:
def read_data():
    year = 0
    while year not in [2018, 2019, 2020]:
        year = int(input("Choose the year that you'd like to process (2018, 2019, 2020): "))
        if year not in [2018, 2019, 2020]:
            print("Invalid year. Please enter 2018, 2019, or 2020.")
        elif year == 2018:
            cargo_df = pd.read_csv('silver//cargo_2018_silver.csv')
            header_df = pd.read_csv('silver//shipment_2018_silver.csv')
            tariff_df = pd.read_csv('silver//tariff_2018_silver.csv')
            shipper_df = pd.read_csv('silver//shipper_2018_silver.csv')
            countries_df = pd.read_csv('silver//countries_2018_silver.csv')
        elif year == 2019:
            cargo_df = pd.read_csv('silver//cargo_2019_silver.csv')
            header_df = pd.read_csv('silver//shipment_2019_silver.csv')
            tariff_df = pd.read_csv('silver//tariff_2019_silver.csv')
            shipper_df = pd.read_csv('silver//shipper_2019_silver.csv')
            countries_df = pd.read_csv('silver//countries_2019_silver.csv')
        elif year == 2020:
            cargo_df = pd.read_csv('silver//cargo_2020_silver.csv')
            header_df = pd.read_csv('silver//shipment_2020_silver.csv')
            tariff_df = pd.read_csv('silver//tariff_2020_silver.csv')
            shipper_df = pd.read_csv('silver//shipper_2020_silver.csv')
            countries_df = pd.read_csv('silver//countries_2020_silver.csv')
    return header_df, shipper_df, cargo_df, tariff_df, countries_df, year

header_df, shipper_df, cargo_df, tariff_df, countries_df, year = read_data()

In [ ]:
# Run this BEFORE anything else to see why joins are failing
print("Header identifier sample:")
print(header_df['identifier'].head())
print("Type:", header_df['identifier'].dtype)

print("\nShipper identifier sample:")
print(shipper_df['identifier'].head())
print("Type:", shipper_df['identifier'].dtype)

# Check how many actually overlap
common = set(header_df['identifier'].astype(str)).intersection(
    set(shipper_df['identifier'].astype(str))
)
print(f"\nMatching identifiers between Header and Shipper: {len(common)}")

In [ ]:
def normalize_identifier(df):
    df = df.copy()
    # Strip whitespace and convert to string for consistent matching
    df['identifier'] = df['identifier'].astype(str).str.strip()
    return df

def build_gold_layer(header_df, shipper_df, cargo_df, year):
    # Normalize all identifier columns to string
    header_df = normalize_identifier(header_df)
    shipper_df = normalize_identifier(shipper_df)
    cargo_df = normalize_identifier(cargo_df)

    # Load and combine tariff across all three years
    tariff_frames = []
    for y in [2018, 2019, 2020]:
        df = pd.read_csv(f'silver//tariff_{y}_silver.csv')
        df['year'] = y
        tariff_frames.append(df)
    tariff_combined = pd.concat(tariff_frames, ignore_index=True)
    tariff_combined = normalize_identifier(tariff_combined)

    # Aggregate tariff FIRST before joining to prevent row explosion
    tariff_agg = tariff_combined.groupby('identifier').agg(
        total_harmonized_weight=('harmonized_weight_kg', 'sum'),
        harmonized_weight_unit=('harmonized_weight_combined', 'first'),
        total_tariff_lines=('identifier', 'count')
    ).reset_index()

    # Join header to shipper
    header_shipper = header_df.merge(shipper_df, on='identifier', how='left')
    print(f"After header-shipper join: {header_shipper.shape[0]} rows")

    # Join aggregated tariff
    gold_df = header_shipper.merge(tariff_agg, on='identifier', how='left')
    print(f"After tariff join: {gold_df.shape[0]} rows")

    # Aggregate by shipper
    gold_summary = gold_df.groupby(
        ['shipper_party_name', 'harmonized_weight_unit']
    ).agg(
        total_harmonized_weight=('total_harmonized_weight', 'sum'),
        total_shipments=('identifier', 'count')
    ).reset_index()

    # Sort by weight descending
    gold_summary = gold_summary.sort_values(
        'total_harmonized_weight', ascending=False
    ).reset_index(drop=True)

    # Add audit columns
    gold_summary['selected_year'] = year
    gold_summary['load_date'] = pd.Timestamp.today().date()
    gold_summary['source_system'] = f'silver_{year}_tariff_2018_2020'

    return gold_summary

In [ ]:
gold_df = build_gold_layer(header_df, shipper_df, cargo_df, year)

print(f"Gold layer rows: {gold_df.shape[0]}")
print(gold_df.head(10))

In [ ]:
os.makedirs('gold', exist_ok=True)
gold_df.to_parquet(f'gold/shipper_gold_{year}.parquet', index=False)
print(f"Saved to gold/shipper_gold_{year}.parquet")

✗ Tariff DataFrame has issues.


,identifier,carrier_code,vessel_name,vessel_country_code,port_of_unlading,estimated_arrival_date,actual_arrival_date,weight_combined,shipper_party_name,city,...,container_number,description_sequence_number,piece_count,description_text,description_sequence_number_tariff,harmonized_number,harmonized_value,harmonized_weight_combined,harmonized_weight_kg,year
0,201801010,NaN,NaN,NaN,NaN,NaN,NaN,NaN,JET FAST COMPANY LIMITED,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2018
1,201801011,NaN,NaN,NaN,NaN,NaN,NaN,NaN,UNION WONDERFUL MACHINERY LTD.,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2018
2,201801012,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"SUMEEKO INDUSTRIES CO.,LTD.",NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2018
3,201801013,NaN,NaN,NaN,NaN,NaN,NaN,NaN,YUTY INDUSTRIES CO. LTD.,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2018
4,201801014,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"BE SOUND CO., LTD.",NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2018
